In [ ]:
!pip install opencv-python numpy

In [10]:
import cv2
import numpy as np
import os

input_folder = "Biscuit Images/input"
output_folder = "Biscuit Images/output"
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):

    if not filename.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    path = os.path.join(input_folder, filename)
    img = cv2.imread(path)

    if img is None:
        continue

    img = cv2.resize(img, (800, 800))
    output = img.copy()


    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    _, thresh = cv2.threshold(
        blur, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    kernel = np.ones((5, 5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, 3)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, 2)

    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    intact = 0
    broken = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)
        if area < 1500:
            continue

        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

    
        circularity = 4 * np.pi * area / (perimeter * perimeter)

        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        solidity = area / hull_area if hull_area != 0 else 0

        x, y, w, h = cv2.boundingRect(cnt)
        extent = area / (w * h)

        
        roughness = perimeter / (2 * np.sqrt(np.pi * area))

    
        is_large = area > 8000

        
        if is_large and (
            circularity > 0.75 and
            solidity > 0.94 and
            roughness < 1.15   
        ):
            label = "Intact Biscuit"
            color = (0, 255, 0)
            intact += 1

        elif is_large and (
            extent > 0.75 and
            solidity > 0.90 and
            roughness < 1.2    
        ):
            label = "Intact Biscuit"
            color = (0, 255, 0)
            intact += 1

        else:
            label = "Broken Biscuit"
            color = (0, 0, 255)
            broken += 1

        
        cv2.drawContours(output, [cnt], -1, color, 3)

        cv2.putText(output, label, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    cv2.putText(output,
                f"Intact: {intact}  Broken: {broken}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 0, 0),
                2)

    out_path = os.path.join(output_folder, "processed_" + filename)
    cv2.imwrite(out_path, output)

    print("Saved:", out_path)

print("DONE")

Saved: Biscuit Images/output\processed_circular biscuit 1.jpg
Saved: Biscuit Images/output\processed_circular biscuit 2.jpg
Saved: Biscuit Images/output\processed_circular biscuit 3.jpg
Saved: Biscuit Images/output\processed_circular biscuit 4.jpg
Saved: Biscuit Images/output\processed_circular biscuit 5.jpg
Saved: Biscuit Images/output\processed_square image 1.jpg
Saved: Biscuit Images/output\processed_square image 2.jpg
Saved: Biscuit Images/output\processed_square image 3.jpg
Saved: Biscuit Images/output\processed_square image 4.jpg
Saved: Biscuit Images/output\processed_square image 5.jpg
DONE
